# Raw Patient Table

Creates a sample patient table in `workspace.health_informatics_assignment.raw_patient_table` containing synthetic patient records with columns that span common PHI categories (demographics, contact info, identifiers, clinical data, and geographic details). This table serves as a practice dataset for Unity Catalog governance features including PHI tagging and column masking based on group membership.

**Inputs:** None (synthetic data generated inline)

**Outputs:** A managed Delta table with 10 sample patient rows

In [0]:
CREATE OR REPLACE TABLE workspace.health_informatics_assignment.raw_patient_table (
  patient_id          BIGINT       COMMENT 'Unique patient identifier',
  mrn                STRING       COMMENT 'Medical record number',
  first_name         STRING       COMMENT 'Patient first name',
  last_name          STRING       COMMENT 'Patient last name',
  date_of_birth      DATE         COMMENT 'Patient date of birth',
  ssn                STRING       COMMENT 'Social Security number',
  phone_number       STRING       COMMENT 'Patient contact phone number',
  email              STRING       COMMENT 'Patient email address',
  street_address     STRING       COMMENT 'Patient street address',
  city               STRING       COMMENT 'Patient city',
  state              STRING       COMMENT 'Patient state',
  zip_code           STRING       COMMENT 'Patient ZIP code',
  insurance_member_id STRING      COMMENT 'Health insurance member ID',
  gender             STRING       COMMENT 'Patient gender',
  race               STRING       COMMENT 'Patient race',
  diagnosis_code     STRING       COMMENT 'ICD-10 diagnosis code',
  diagnosis_description STRING    COMMENT 'Diagnosis description',
  admission_date     DATE         COMMENT 'Hospital admission date',
  attending_physician STRING      COMMENT 'Name of attending physician'
);

-- Add table-level comment
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table
SET TBLPROPERTIES ('comment' = 'Synthetic patient records for practicing PHI tagging and column masking');

In [0]:
INSERT INTO workspace.health_informatics_assignment.raw_patient_table
VALUES
  (1,  'MRN-100001', 'James',   'Smith',     DATE '1985-03-15', '123-45-6789', '555-0101', 'james.smith@email.com',     '123 Oak St',     'Philadelphia', 'PA', '19101', 'INS-9001', 'Male',   'White',    'I10',   'Essential hypertension',                 DATE '2024-01-15', 'Dr. Alice Chen'),
  (2,  'MRN-100002', 'Maria',   'Garcia',    DATE '1990-07-22', '234-56-7890', '555-0102', 'maria.garcia@email.com',    '456 Pine Ave',    'Pittsburgh',   'PA', '15201', 'INS-9002', 'Female', 'Hispanic', 'E11.9', 'Type 2 diabetes mellitus',                DATE '2024-02-03', 'Dr. Brian Lee'),
  (3,  'MRN-100003', 'Robert',  'Johnson',   DATE '1978-11-30', '345-67-8901', '555-0103', 'robert.johnson@email.com', '789 Maple Dr',   'Philadelphia', 'PA', '19102', 'INS-9003', 'Male',   'Black',    'J44.9', 'COPD unspecified',                       DATE '2024-01-28', 'Dr. Alice Chen'),
  (4,  'MRN-100004', 'Patricia', 'Williams', DATE '1995-04-10', '456-78-9012', '555-0104', 'patricia.w@email.com',     '321 Cedar Ln',   'Harrisburg',   'PA', '17101', 'INS-9004', 'Female', 'White',    'F41.1', 'Generalized anxiety disorder',           DATE '2024-03-05', 'Dr. Carol Park'),
  (5,  'MRN-100005', 'John',    'Brown',     DATE '1970-09-18', '567-89-0123', '555-0105', 'john.brown@email.com',     '654 Birch Rd',   'Allentown',    'PA', '18101', 'INS-9005', 'Male',   'White',    'I21.4', 'Non-ST elevation myocardial infarction', DATE '2024-02-20', 'Dr. David Kim'),
  (6,  'MRN-100006', 'Linda',   'Davis',     DATE '1988-12-05', '678-90-1234', '555-0106', 'linda.davis@email.com',    '987 Walnut St',  'Philadelphia', 'PA', '19103', 'INS-9006', 'Female', 'Asian',    'N18.3', 'Chronic kidney disease stage 3',        DATE '2024-03-12', 'Dr. Brian Lee'),
  (7,  'MRN-100007', 'Michael', 'Miller',    DATE '1982-06-25', '789-01-2345', '555-0107', 'michael.m@email.com',     '147 Elm Ct',     'Pittsburgh',   'PA', '15202', 'INS-9007', 'Male',   'White',    'M54.5', 'Low back pain',                          DATE '2024-01-08', 'Dr. Carol Park'),
  (8,  'MRN-100008', 'Sarah',   'Wilson',    DATE '1993-02-14', '890-12-3456', '555-0108', 'sarah.wilson@email.com',   '258 Spruce Way','Erie',         'PA', '16501', 'INS-9008', 'Female', 'Black',    'O80',   'Encounter for full-term uncomplicated delivery', DATE '2024-03-18', 'Dr. David Kim'),
  (9,  'MRN-100009', 'David',   'Taylor',    DATE '1975-08-03', '901-23-4567', '555-0109', 'david.taylor@email.com',  '369 Aspen Blvd', 'Scranton',     'PA', '18501', 'INS-9009', 'Male',   'Hispanic', 'J45.909','Asthma unspecified',                     DATE '2024-02-10', 'Dr. Alice Chen'),
  (10, 'MRN-100010', 'Jennifer','Anderson', DATE '1998-10-20', '012-34-5678', '555-0110', 'jennifer.a@email.com',    '741 Willow Ct',  'Philadelphia', 'PA', '19104', 'INS-9010', 'Female', 'White',    'R51.9', 'Headache unspecified',                   DATE '2024-03-22', 'Dr. Carol Park');

In [0]:
SELECT * FROM workspace.health_informatics_assignment.raw_patient_table;

In [0]:
-- Apply custom governed tags (phi, phi_type) to classify PHI columns
-- Tag key -> column mapping:
--   phi = true                          -> table (entire table contains PHI)
--   phi_type = direct_identifier        -> mrn, first_name, last_name, ssn, date_of_birth, phone_number, email, insurance_member_id
--   phi_type = quasi_identifier          -> street_address, city, state, zip_code
--   phi_type = sensitive_demographic     -> race
--   phi_type = clinical_data             -> diagnosis_code, diagnosis_description, admission_date

-- Table-level: mark the entire table as containing PHI
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table
SET TAGS ('phi' = 'true');

-- Direct identifiers (name, SSN, DOB, MRN, contact, insurance ID)
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN mrn SET TAGS ('phi' = 'true', 'phi_type' = 'direct_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN first_name SET TAGS ('phi' = 'true', 'phi_type' = 'direct_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN last_name SET TAGS ('phi' = 'true', 'phi_type' = 'direct_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN ssn SET TAGS ('phi' = 'true', 'phi_type' = 'direct_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN date_of_birth SET TAGS ('phi' = 'true', 'phi_type' = 'direct_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN phone_number SET TAGS ('phi' = 'true', 'phi_type' = 'direct_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN email SET TAGS ('phi' = 'true', 'phi_type' = 'direct_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN insurance_member_id SET TAGS ('phi' = 'true', 'phi_type' = 'direct_identifier');

-- Quasi-identifiers (geographic location)
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN street_address SET TAGS ('phi' = 'true', 'phi_type' = 'quasi_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN city SET TAGS ('phi' = 'true', 'phi_type' = 'quasi_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN state SET TAGS ('phi' = 'true', 'phi_type' = 'quasi_identifier');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN zip_code SET TAGS ('phi' = 'true', 'phi_type' = 'quasi_identifier');

-- Sensitive demographic
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN race SET TAGS ('phi' = 'true', 'phi_type' = 'sensitive_demographic');

-- Clinical data
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN diagnosis_code SET TAGS ('phi' = 'true', 'phi_type' = 'clinical_data');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN diagnosis_description SET TAGS ('phi' = 'true', 'phi_type' = 'clinical_data');
ALTER TABLE workspace.health_informatics_assignment.raw_patient_table ALTER COLUMN admission_date SET TAGS ('phi' = 'true', 'phi_type' = 'clinical_data');

In [0]:
-- Show all column-level governed tags applied to the table
SELECT
  column_name,
  tag_name,
  tag_value
FROM system.information_schema.column_tags
WHERE catalog_name = 'workspace'
  AND schema_name  = 'health_informatics_assignment'
  AND table_name   = 'raw_patient_table'
ORDER BY column_name, tag_name;